# CMU-MOSEI Detectorv2 visual prompt — train split

This Kaggle notebook extracts frozen Qwen2.5-VL-3B video + transcript embeddings with Detectorv2 speaker-face measurements in the prompt. It covers the **train** split only, in independent 500 clip chunks. Each included clip has one target: its unique highest positive emotion score, or neutral when all six scores are zero. Tied highest positive scores are excluded. The prompt asks Qwen to choose one of these seven classes.

Attach the current `pushkarsingh2005/benchmark-code` and `pushkarsingh2005/cmu-mosei-raw-videos-and-csvs` datasets. Enable a GPU and Internet for the model and Py-Feat weights. Refresh `benchmark-code` from the repository containing `extract_mosei_detectorv2_prompt.py` and the updated `prepare_mosei.py` before running.


## 1. Inputs and chunk selection

Set `START_CHUNK` to a zero-based chunk number and `NUM_CHUNKS` to the number of consecutive chunks for this Kaggle session. Run chunk 0 first; continue with later chunks in new sessions as needed. To carry completed chunks into a new session, attach a prior notebook output as a Kaggle input and set `PREVIOUS_CHUNKS_DIR` to its `chunks/<split>` directory. Never mix chunks from different extractor settings or code versions.


In [ ]:
from pathlib import Path
import shutil

CODE = Path('/kaggle/input/datasets/pushkarsingh2005/benchmark-code')
MOSEI_ROOT = Path('/kaggle/input/datasets/pushkarsingh2005/cmu-mosei-raw-videos-and-csvs')
RAW_DIR = MOSEI_ROOT / 'Raw' / 'Raw'
RUN_ROOT = Path('/kaggle/working/detectorv2_prompt_mosei_singlemax')
SPLIT = 'train'
EXPECTED_ROWS = 14273
CHUNK_SIZE = 500
START_CHUNK = 0
NUM_CHUNKS = 1
PREVIOUS_CHUNKS_DIR = None  # e.g. Path('/kaggle/input/datasets/<owner>/<prior-output>/chunks/train')

PREPARE = CODE / 'scripts/data/prepare_mosei.py'
EXTRACTOR = CODE / 'scripts/extraction/extract_mosei_detectorv2_prompt.py'
MERGER = CODE / 'scripts/extraction/merge_embedding_chunks.py'
INDEX_DIR = RUN_ROOT / 'indexes'
CHUNKS_DIR = RUN_ROOT / 'chunks' / SPLIT
OUTPUT = RUN_ROOT / f'mosei_{SPLIT}.pt'
print('Checking Kaggle inputs for', SPLIT, flush=True)
for path in (CODE, MOSEI_ROOT, RAW_DIR, PREPARE, EXTRACTOR, MERGER):
    print('Checking:', path, flush=True)
    assert path.exists(), f'Missing {path}. Check attached Kaggle inputs and refresh benchmark-code.'
assert 'single maximum-score emotion; tied maxima excluded; all-zero neutral' in EXTRACTOR.read_text(), 'Attached benchmark-code is too old.'
assert START_CHUNK >= 0 and NUM_CHUNKS > 0
INDEX_DIR.mkdir(parents=True, exist_ok=True)
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

# Set this explicitly if the aligned CSV is attached at another location.
ALIGNED_CSV = None
if ALIGNED_CSV is None:
    csv_candidates = (
        MOSEI_ROOT / 'mosei_aligned.csv',
        MOSEI_ROOT / 'Raw' / 'mosei_aligned.csv',
        MOSEI_ROOT / 'Raw' / 'Raw' / 'mosei_aligned.csv',
        CODE / 'dataset' / 'CMU-MOSEI' / 'csv' / 'mosei_aligned.csv',
    )
    print('Checking aligned CSV at known locations', flush=True)
    ALIGNED_CSV = next((path for path in csv_candidates if path.is_file()), None)
    assert ALIGNED_CSV is not None, (
        'mosei_aligned.csv was not found at the listed locations. '
        'Set ALIGNED_CSV to its exact Kaggle path in this cell. '
        f'Checked: {[str(path) for path in csv_candidates]}'
    )
ALIGNED_CSV = Path(ALIGNED_CSV)
assert ALIGNED_CSV.is_file(), f'Missing aligned CSV: {ALIGNED_CSV}'
print('CSV:', ALIGNED_CSV, 'Videos:', RAW_DIR, flush=True)
print('Split:', SPLIT, 'Chunks:', START_CHUNK, 'through', START_CHUNK + NUM_CHUNKS - 1, flush=True)


## 2. Dependencies and GPU

The extraction script runs in a fresh Python process after installation. Py-Feat Detectorv2 temporarily disables cuDNN for its own call; Qwen retains its normal backend.


In [ ]:
!python -m pip uninstall -y -q gradio diffusers
!python -m pip install -q "py-feat==2.1.5" "transformers==4.57.1" "huggingface-hub>=0.34,<1" "Pillow>=10,<13" "qwen-vl-utils[decord]>=0.0.14" decord accelerate pandas scikit-learn


In [ ]:
from importlib.metadata import version
from packaging.version import Version
import torch

assert version('py-feat') == '2.1.5'
assert version('transformers') == '4.57.1'
assert Version('0.34') <= Version(version('huggingface-hub')) < Version('1')
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
for i in range(torch.cuda.device_count()):
    gpu = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {gpu.name}, {gpu.total_memory / 2**30:.1f} GiB')


## 3. Build the MOSEI index

The Kaggle video folders are under `Raw/Raw/<video_id>`. The updated indexer accepts this exact directory and the aligned CSV independently. It validates IDs, split isolation, labels, timestamps, and clip existence. This run excludes tied positive maxima and gives all-zero clips the neutral class.


In [ ]:
import subprocess
import sys
import pandas as pd

subprocess.run([sys.executable, '-u', str(PREPARE),
                '--aligned-csv', str(ALIGNED_CSV), '--raw-dir', str(RAW_DIR),
                '--output-dir', str(INDEX_DIR), '--target-mode', 'single_max',
                '--tie-policy', 'exclude', '--zero-policy', 'neutral'], check=True)
INDEX = INDEX_DIR / f'mosei_{SPLIT}_index.csv'
index = pd.read_csv(INDEX)
assert len(index) == EXPECTED_ROWS, f'{SPLIT}: expected {EXPECTED_ROWS}, found {len(index)}'
assert index.sample_id.is_unique and index.split.eq(SPLIT).all()
assert index.video_exists.eq(True).all()
assert all(Path(path).is_file() for path in index.video_path)
assert set(index.label_status) <= {'unique_max', 'all_zero'}
assert set(index.emotion) <= {'happy','sad','anger','surprise','disgust','fear','neutral'}
assert index.emotion_id.between(0, 6).all()
print(SPLIT, len(index), 'verified clips;', (len(index) + CHUNK_SIZE - 1) // CHUNK_SIZE, 'chunks')


## 4. Extract the selected 500 clip chunks

Each chunk has its own `.pt` checkpoint. Rerunning this cell resumes saved successes and retries errors. A chunk with errors is never treated as complete. The extractor saves every 25 attempted clips and stops on repeated systematic failures. `PREVIOUS_CHUNKS_DIR` copies earlier output into the writable working directory before extraction.


In [ ]:
import math
import hashlib

TOTAL_CHUNKS = math.ceil(len(index) / CHUNK_SIZE)
assert START_CHUNK < TOTAL_CHUNKS
assert START_CHUNK + NUM_CHUNKS <= TOTAL_CHUNKS, 'Selected range exceeds the split.'
if PREVIOUS_CHUNKS_DIR is not None:
    previous = Path(PREVIOUS_CHUNKS_DIR)
    assert previous.is_dir(), f'Missing prior chunk directory: {previous}'
    for source in previous.glob(f'{SPLIT}_*.pt'):
        target = CHUNKS_DIR / source.name
        if not target.exists():
            shutil.copy2(source, target)

def chunk_path(chunk_number):
    start = chunk_number * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, len(index))
    return CHUNKS_DIR / f'{SPLIT}_{start:05d}_{end:05d}.pt'

def check_chunk(path, start, end):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    expected = set(index.sample_id.iloc[start:end].astype(str))
    ids = list(map(str, payload['sample_ids']))
    errors = payload.get('errors', [])
    assert len(ids) == len(set(ids)) and set(ids) <= expected
    assert payload['embeddings'].shape[0] == len(ids)
    assert tuple(payload['labels'].shape) == (len(ids),)
    expected_labels = torch.tensor(index.set_index('sample_id').loc[ids, 'emotion_id'].to_numpy(), dtype=torch.long)
    assert torch.equal(payload['labels'].long(), expected_labels), 'Chunk labels differ from this index.'
    assert len(payload['metadata']) == len(ids)
    assert tuple(payload['label_names']) == ('happy','sad','anger','surprise','disgust','fear','neutral')
    assert payload['config']['start'] == start and payload['config']['limit'] == end - start
    config = payload['config']
    assert config['detector_cudnn_enabled'] is False
    assert config['model_id'] == 'Qwen/Qwen2.5-VL-3B-Instruct'
    assert (config['fps'], config['min_frames'], config['max_frames'], config['frame_size'], config['face_threshold']) == (6.0, 4, 64, 224, 0.7)
    assert config['target'] == 'single maximum-score emotion; tied maxima excluded; all-zero neutral'
    assert config['extractor_sha256'] == hashlib.sha256(EXTRACTOR.read_bytes()).hexdigest(), 'Chunk came from a different extractor revision.'
    assert torch.isfinite(payload['embeddings']).all()
    error_ids = {str(row['sample_id']) for row in errors}
    assert set(ids).isdisjoint(error_ids) and error_ids <= expected
    return payload, len(ids) == end - start and not errors and set(ids) == expected



In [ ]:
for chunk_number in range(START_CHUNK, START_CHUNK + NUM_CHUNKS):
    start = chunk_number * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, len(index))
    path = chunk_path(chunk_number)
    if path.exists():
        _, complete = check_chunk(path, start, end)
        if complete:
            print('Already complete:', path)
            continue
    command = [sys.executable, '-u', str(EXTRACTOR),
               '--index-csv', str(INDEX), '--output-pt', str(path),
               '--start', str(start), '--limit', str(end - start),
               '--fps', '6', '--min-frames', '4', '--max-frames', '64',
               '--frame-size', '224', '--face-threshold', '0.7', '--save-every', '25']
    if path.exists():
        command += ['--resume', '--retry-errors']
    print('Running chunk', chunk_number, f'[{start}:{end}]', flush=True)
    subprocess.run(command, check=True)
    payload, complete = check_chunk(path, start, end)
    print(path.name, len(payload['sample_ids']), 'embeddings;', len(payload['errors']), 'errors')
    assert complete, 'Chunk has missing clips or extraction errors. Inspect errors and rerun this cell.'


## 5. Inspect results and merge when all chunks are present

The merge runs only after every expected chunk is present and error free. For the full train split, gather prior session outputs through `PREVIOUS_CHUNKS_DIR` and run this section again. Then attach the merged train, valid, and test outputs to a separate seven-class MLP training job. Select its checkpoint on valid and score test once.


In [ ]:
from collections import Counter
from IPython.display import FileLink, display

available = []
for chunk_number in range(TOTAL_CHUNKS):
    start = chunk_number * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, len(index))
    path = chunk_path(chunk_number)
    if path.exists():
        payload, complete = check_chunk(path, start, end)
        assert complete, f'Incomplete chunk: {path}'
        available.append(path)
print(f'Complete chunks: {len(available)}/{TOTAL_CHUNKS}')
for chunk_number in range(START_CHUNK, START_CHUNK + NUM_CHUNKS):
    path = chunk_path(chunk_number)
    payload = torch.load(path, map_location='cpu', weights_only=False)
    print(path.name, 'face statuses:', Counter(m['speaker_face']['status'] for m in payload['metadata']))
    if payload['metadata']:
        print('Example prompt:', payload['metadata'][0]['prompt'][:1200])
    display(FileLink(str(path)))
if len(available) == TOTAL_CHUNKS:
    subprocess.run([sys.executable, '-u', str(MERGER),
                    '--chunks-dir', str(CHUNKS_DIR), '--pattern', f'{SPLIT}_*.pt',
                    '--output-pt', str(OUTPUT)], check=True)
    merged = torch.load(OUTPUT, map_location='cpu', weights_only=False)
    assert len(merged['sample_ids']) == len(index) == len(set(merged['sample_ids']))
    assert set(merged['sample_ids']) == set(index.sample_id.astype(str))
    assert tuple(merged['labels'].shape) == (len(index),)
    assert torch.equal(merged['labels'].long(), torch.tensor(index.set_index('sample_id').loc[merged['sample_ids'], 'emotion_id'].to_numpy(), dtype=torch.long))
    display(FileLink(str(OUTPUT)))
else:
    print('Next chunk:', next(i for i in range(TOTAL_CHUNKS) if not chunk_path(i).exists()))
